### 1. Setup:

In [1]:
import getpass
import os

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = getpass.getpass()

In [20]:
from langchain_ollama import ChatOllama

# ChatOllama supports tool binding for agent use
model = ChatOllama(
    model="mistral:latest",
    temperature=0.7,
    base_url="http://localhost:11434"
)

### Indexing:

In [3]:
import getpass
import os

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [4]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

### 1. Indexing:

In [21]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

# Only keep post title, headers, and content from the full HTML.
bs4_strainer = bs4.SoupStrainer(class_=("post-title", "post-header", "post-content"))
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs={"parse_only": bs4_strainer},
)
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 43047


In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=200,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 63 sub-documents.


In [23]:
document_ids = vector_store.add_documents(documents=all_splits)

print(document_ids[:3])

['f8f000ef-6e08-4665-b16f-f0d3de11a1bd', '1d72fd09-d5ad-47da-82e5-3d7551dab477', '834af7be-fddc-4043-a497-4bdc9757d9cd']


### 3. Retrieval and Generation:

In [24]:
from langchain.tools import tool

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

In [25]:
from langchain.agents import create_agent


tools = [retrieve_context]
# If desired, specify custom instructions
prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)
agent = create_agent(model, tools, system_prompt=prompt)

In [26]:
query = (
    "What is the standard method for Task Decomposition?\n\n"
    "Once you get the answer, look up common extensions of that method."
)

for event in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

What is the standard method for Task Decomposition?

Once you get the answer, look up common extensions of that method.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (eacc1498-48c1-45a0-9eab-9111b73655eb)
 Call ID: eacc1498-48c1-45a0-9eab-9111b73655eb
  Args:
    query: What is the standard method for Task Decomposition?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (eacc1498-48c1-45a0-9eab-9111b73655eb)
 Call ID: eacc1498-48c1-45a0-9eab-9111b73655eb
  Args:
    query: What is the standard method for Task Decomposition?
================================= Tool Message =================================
Name: retrieve_context

Source: {'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 2578}
Content: Task decomposition can be done (1) by LLM with sim

In [28]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def prompt_with_context(request: ModelRequest) -> str:
    """Inject context into state messages."""
    last_query = request.state["messages"][-1].text
    retrieved_docs = vector_store.similarity_search(last_query)

    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    system_message = (
        "You are a helpful assistant. Use the following context in your response:"
        f"\n\n{docs_content}"
    )

    return system_message


agent = create_agent(model, tools=[], middleware=[prompt_with_context])

In [29]:
query = "What is task decomposition?"
for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is task decomposition?
================================== Ai Message ==================================

 Task decomposition refers to the process of breaking down complex tasks into smaller, manageable subtasks. This method allows an agent to plan ahead and execute the steps required to achieve the overall goal. In the context provided, two approaches for task decomposition are discussed:

1. LLM-based task decomposition involves using simple prompting like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?" for various tasks. This method can also employ task-specific instructions, such as "Write a story outline" for writing a novel, or incorporate human inputs when necessary.

2. LLM+P (Liu et al., 2023) is another approach that outsources long-horizon planning to an external classical planner using the Planning Domain Definition Language (PDDL). The process involves translating the 